In [1]:
from dotenv import load_dotenv
from groq import Groq
import os

In [2]:
from langchain_core.prompts import PromptTemplate

In [3]:
from pathlib import Path

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API key loaded:", GROQ_API_KEY is not None)

API key loaded: True


In [5]:
client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [6]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "What is Ancient Egypt?"
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

**Ancient Egypt** was one of the world’s earliest and most enduring civilizations, flourishing along the Nile River in northeastern Africa from roughly 3100 BCE to 30 BCE. It is best known for its monumental architecture, sophisticated writing system, and complex religious beliefs, but its influence extended far beyond those iconic symbols.

| Aspect | Key Points |
|--------|------------|
| **Geography & Environment** | The Nile’s predictable flooding created fertile floodplains, enabling stable agriculture. The desert to the east and west provided natural protection. |
| **Timeline** | • **Early Dynastic Period** (c. 3100–2686 BCE) – unification under the first pharaoh, Narmer (or Menes).  <br>• **Old Kingdom** (2686–2181 BCE) – “Age of the Pyramids” (Great Pyramid of Giza, Step Pyramid of Saqqara).  <br>• **First Intermediate Period** (c. 2181–2055 BCE) – political fragmentation.  <br>• **Middle Kingdom** (c. 2055–1650 BCE) – reunification, literary flourishing.  <br>• **Second Inter

### Generating the Answer

The retrieved context and the user's question are passed to the LLM through the prompt template. The model generates an answer using only the retrieved information and returns the response along with its source information.


In [7]:
persist_directory = Path("../Data/chroma_db")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma(
    persist_directory=str(persist_directory),
    embedding_function=embedding_model
)

print("Vector store loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store loaded successfully.


In [8]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant answering questions about Ancient Egypt.

Use ONLY the information provided in the context below.
Do not use outside knowledge.
If the answer cannot be found in the context, say:
"I could not find the answer in the provided document."

At the end of your answer, provide the source chapter and chunk ID.

Context:
{context}

Question:
{question}

Answer:
"""
)

print(prompt_template)

input_variables=['context', 'question'] input_types={} partial_variables={} template='\nYou are a helpful assistant answering questions about Ancient Egypt.\n\nUse ONLY the information provided in the context below.\nDo not use outside knowledge.\nIf the answer cannot be found in the context, say:\n"I could not find the answer in the provided document."\n\nAt the end of your answer, provide the source chapter and chunk ID.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n'


In [9]:
def rag_answer(question, k=2):
    
    # Retrieve relevant chunks
    results = vectordb.similarity_search(question, k=k)

    # Build context
    context_parts = []
    sources = []

    for i, doc in enumerate(results, start=1):

        source = doc.metadata.get("source", "Unknown")

        context_parts.append(
            f"[Chunk {i} | Source: {source}]\n"
            f"{doc.page_content}"
        )

        sources.append(f"Chunk {i} — {source}")

    context = "\n\n---\n\n".join(context_parts)

    # Build prompt
    prompt = prompt_template.format(
        context=context,
        question=question
    )

    # Call LLM
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    return answer, sources

test the questions

In [10]:
questions = [
    "Who was Cleopatra?",
    "What was Ancient Egypt?",
    "Where was Ancient Egypt located?",
    "What was the importance of the Nile River to Ancient Egypt?",
    "What were the pyramids used for?",
    "Who built the pyramids?",
    "What was the role of pharaohs in Ancient Egypt?",
    "What were some important achievements of Ancient Egyptian civilization?",
    "What was Ancient Egyptian writing called?",
    "What were the main religious beliefs of Ancient Egyptians?"
]

In [11]:
for i, question in enumerate(questions, start=1):

    answer, sources = rag_answer(question)

    print("=" * 80)
    print(f"Question {i}: {question}")

    print("\nAnswer:")
    print(answer)



    print()

Question 1: Who was Cleopatra?

Answer:
I could not find the answer in the provided document.  
Source: ancient_egypt.html, Chunk 1

Question 2: What was Ancient Egypt?

Answer:
Ancient Egypt was the valley and plain watered by the Nile, stretching for nearly seven hundred miles along the river’s course.  

Source: Chunk 1 (ancient_egypt.html)

Question 3: Where was Ancient Egypt located?

Answer:
Ancient Egypt was located in the valley and plain watered by the Nile, stretching for nearly seven hundred miles along the river’s course. This area lies in the northeastern corner of Africa, bounded on two sides by the Mediterranean and the Red Sea, but the true extent of Egypt is the Nile valley itself rather than the larger rectangular area shown on maps.  

Source: ancient_egypt.html, Chunk 1

Question 4: What was the importance of the Nile River to Ancient Egypt?

Answer:
The Nile was considered “useful, beneficent, and indeed essential to the existence of Egypt.” It was the lifeblood th

## 2.6 Evaluation

The RAG system was evaluated using 10 test questions. For each question, we checked whether the retrieved context was relevant, whether the answer was grounded in the retrieved context, and whether the final answer was correct.

| # | Question | Retrieved Source | Context Relevant? | Grounded? | Correct? |
|---|---|---|---|---|---|
| 1 | Who was Cleopatra? | ancient_egypt.html | No | Yes | Yes |
| 2 | What was Ancient Egypt? | ancient_egypt.html | Yes | Yes | Yes |
| 3 | Where was Ancient Egypt located? | ancient_egypt.html | Yes | Yes | Yes |
| 4 | What was the importance of the Nile River to Ancient Egypt? | ancient_egypt.html | Yes | Yes | Yes |
| 5 | What were the pyramids used for? | ancient_egypt.html | Yes | Yes | Yes |
| 6 | Who built the pyramids? | ancient_egypt.html | Yes | Yes | Yes |
| 7 | What was the role of pharaohs in Ancient Egypt? | ancient_egypt.html | Yes | Yes | Yes |
| 8 | What were some important achievements of Ancient Egyptian civilization? | ancient_egypt.html | Yes | Yes | Yes |
| 9 | What was Ancient Egyptian writing called? | ancient_egypt.html | Yes | Yes | Yes |
| 10 | What were the main religious beliefs of Ancient Egyptians? | ancient_egypt.html | Yes | Yes | Yes |

### Failure Cases and Mitigation

The main failure cases occurred when the question was not directly covered by the retrieved document. In these cases, the system was instructed not to generate unsupported information and instead respond that there was not enough information in the provided documents. Using a restricted prompt that allows the LLM to answer only from the retrieved context helped reduce hallucinations. The retrieval results were also inspected to ensure that the selected chunks were relevant to the user's question.